# 체험 1: 새 동료를 소개합니다

**체험 1** > 체험 2 > 체험 3 > 자유 실습

---

여러분 옆에 코딩을 잘하는 신입이 앉아있다고 상상해보세요.
한국어를 알아듣고, 시키면 뭐든 해봅니다. 대신 가끔 실수도 합니다.

일단 간단한 것부터 시켜봅시다. 진짜 말이 통하는지.

> **먼저 아래 [준비] 셀을 실행하세요.** 재생 버튼(>) 또는 Shift+Enter

---

## Colab AI 사용법

| 순서 | 동작 |
|:----:|------|
| 1 | 빈 코드 셀 클릭 (또는 상단 `+ Code`) |
| 2 | **Generate** 버튼 클릭 |
| 3 | 한글로 입력 |
| 4 | **Insert** --> **Shift+Enter** |

---

## Step 1: 인사부터 해봅시다

빈 코드 셀에서 Generate를 누르고 입력해보세요:

```
안녕하세요 출력해줘
```

되셨으면 하나 더:

```
1부터 10까지 합을 구해줘
```

하나만 더:

```
구구단 7단을 출력해줘
```

> 말이 통하죠?
> 이 신입은 24시간 일하고, 불평도 없고, 시키면 1초 만에 합니다.
> 대신 **검증은 여러분 몫**입니다. AI가 짠 코드가 맞는지 결과를 꼭 확인하세요.

---

## Step 2: 라면 레시피를 AI에게 맡기다

아까 슬라이드에서 라면으로 Y=f(X)를 배웠죠.
라면 실험 데이터 40개와 AI 모델(`ramen_model`)이 준비되어 있습니다.

여러분이 **라면 공장의 품질 담당자**라고 상상해보세요. 신입에게 시킵니다:

```
ramen_model로 물 400, 불 5, 스프 1.5, 시간 5.0일 때 염도와 면발굵기를 예측해줘
```

숫자를 바꿔서 여러 레시피를 시도해보세요:

```
물 550, 불 3, 스프 0.5, 시간 3.5일 때는?
```

> **[미션]** 염도 0.8 이하 + 면발굵기 5.0 이상이 되는 레시피를 찾아보세요.
> 찾으셨으면 공유 시트 "라면" 칸에 적어주세요.

---

3분 만에 AI에게 여러 번 시켰습니다. 코드를 한 줄도 안 썼습니다.
라면도 되니까, 공장 데이터도 될 겁니다.

다음 파일: `2_품질예측.ipynb`

---

**실수 복구:** 뭔가 꼬이면 상단 메뉴 **런타임 > 런타임 다시 시작** --> [준비] 셀 다시 실행


In [ ]:
#@title **[준비] 실행 버튼(>)만 눌러주세요** (코드를 읽을 필요 없습니다)
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

_ws = {'description_width':'120px'}
_wl = widgets.Layout(width='500px')

ramen = pd.DataFrame({
    '물의양':   [590,600,580,600,450,500,580,580,450,450,500,500,500,450,450,450,400,400,400,400,
                 580,590,575,600,450,490,585,585,440,450,500,490,510,455,440,460,390,420,350,420],
    '불의세기': [3,3,3,3,5,5,5,5,5,5,5,5,5,6,5,5,5,6,6,6,3,3,3,3,5,5,5,5,5,5,5,5,5,6,5,5,5,6,6,6],
    '스프의양': [0.5,0.5,0.4,0.6,1.4,0.9,0.7,0.6,1.8,1.4,0.8,0.7,0.9,1.1,1.2,1.5,2.1,1.9,1.5,2.1,
                 0.5,0.5,0.4,0.6,1.4,0.9,0.7,0.6,1.8,1.4,0.8,0.7,0.9,1.1,1.2,1.5,2.1,1.9,1.5,2.1],
    '조리시간': [3.5,3.5,3.7,3.7,5.4,4.5,4.1,4.1,6.1,5.4,4.5,4.7,4.7,5.1,6.1,5.2,6.4,6.4,7.1,7.1,
                 3.5,3.5,3.7,3.7,5.4,4.5,4.1,4.1,6.1,5.4,4.5,4.7,4.7,5.1,6.1,5.2,6.4,6.4,7.1,7.1],
    '염도':     [0.65,0.65,0.67,0.66,1.15,0.82,0.75,0.74,1.20,1.15,0.80,0.85,0.85,1.10,1.10,1.15,1.35,1.35,1.20,1.50,
                 0.65,0.66,0.69,0.71,1.22,0.83,0.72,0.73,1.15,1.15,0.80,0.84,0.81,1.25,1.05,1.26,1.42,1.25,1.40,1.25],
    '면발굵기': [2.5,2.5,2.7,2.8,6.3,5.5,5.0,5.0,6.5,6.4,5.6,6.2,6.2,6.3,6.8,6.4,7.1,7.2,8.0,7.9,
                 2.4,2.6,2.6,2.8,6.4,5.4,5.1,4.9,6.0,6.0,6.4,6.3,6.1,6.2,6.7,6.5,7.2,7.1,8.2,7.9],
})
ramen_model = RandomForestRegressor(n_estimators=100, random_state=42)
ramen_model.fit(ramen[['물의양','불의세기','스프의양','조리시간']], ramen[['염도','면발굵기']])

def _ramen_widget():
    w = widgets.IntSlider(value=550,min=350,max=600,step=10,description='물의양(mL):',style=_ws,layout=_wl)
    f = widgets.IntSlider(value=3,min=3,max=6,step=1,description='불의세기:',style=_ws,layout=_wl)
    s = widgets.FloatSlider(value=0.5,min=0.4,max=2.1,step=0.1,description='스프의양(봉):',style=_ws,layout=_wl)
    t = widgets.FloatSlider(value=3.5,min=3.5,max=7.1,step=0.1,description='조리시간(분):',style=_ws,layout=_wl)
    out = widgets.HTML()
    def up(*_):
        p = ramen_model.predict([[w.value,f.value,s.value,t.value]])[0]
        out.value = ('<div style="margin:12px 0;padding:12px 16px;background:#f0f7ff;border-left:4px solid #3B82F6;border-radius:6px;font-size:15px">'
            f'<b>예측 염도:</b> {p[0]:.2f} &nbsp;&nbsp;&nbsp; <b>예측 면발굵기:</b> {p[1]:.1f}</div>')
    for x in [w,f,s,t]: x.observe(up,'value')
    up()
    return widgets.VBox([w,f,s,t,out])
ramen_ui = _ramen_widget()

print('=' * 50)
print('  [준비] 완료!')
print('=' * 50)
print(f'  라면 데이터: {len(ramen)}회 실험')
print(f'  라면 모델: ramen_model (Random Forest)')
print()
print('  사용 가능한 변수:')
print('    ramen       -- 라면 실험 데이터 (DataFrame)')
print('    ramen_model -- 라면 예측 모델')
print('      입력: [물의양, 불의세기, 스프의양, 조리시간]')
print('      출력: [염도, 면발굵기]')
print()
print('  위로 스크롤하여 Colab AI에게 시켜보세요!')


---

### [선택] 라면 슬라이더 위젯

Colab AI 대신 슬라이더로 체험하고 싶으면 아래 셀을 실행하세요.


In [ ]:
ramen_ui